# 🎤 RVC v2: Персональный голос для песен (Сергей Трофимов — Аты-баты)

**Инструкция: Нажмите `Среда выполнения` -> `Выполнить все` (Runtime -> Run all).**

Пайплайн выполнится полностью автоматически за **~10 минут**:
1. Быстрая подготовка среды Applio, Demucs и весов на GPU Tesla T4.
2. Подтягивается ваш чистый 9-минутный голос (`Testr2.wav`) и трек `Сергей Трофимов - Аты-баты`.
3. Извлекаются частоты связок (алгоритм RMVPE) и обучается модель (150 эпох, ~6-7 мин).
4. Нейросеть Demucs разделяет песню на музыку и вокал.
5. RVC заменяет оригинальный голос Трофимова на **ваш персональный голос** с сохранением всех нот и интонаций!
6. Готовый трек проигрывается прямо в блокноте и скачивается на устройство.

## Шаг 1. Быстрая установка Applio, Demucs и предобученных моделей

In [ ]:
!nvidia-smi

import os, urllib.request, subprocess

%cd /content
if not os.path.exists("/content/Applio"):
    !git clone --depth 1 https://github.com/IAHispano/Applio.git /content/Applio
%cd /content/Applio

# Системные пакеты и ускоренный загрузчик aria2
!apt-get update -y -qq && apt-get install -y -qq aria2 ffmpeg portaudio19-dev

# Установка зависимостей напрямую в Colab окружение
# Установка библиотек RVC и Demucs с гарантированной поддержкой CUDA
!pip install -q faiss-cpu torchcrepe torchfcpe stftpitchshift soxr einops demucs
!pip install -q --upgrade --force-reinstall torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

# Создание папок для моделей
os.makedirs("rvc/models/predictors", exist_ok=True)
os.makedirs("rvc/models/embedders/contentvec", exist_ok=True)
os.makedirs("rvc/models/pretraineds/hifi-gan", exist_ok=True)
os.makedirs("assets/datasets/MyVoice", exist_ok=True)

# Параллельная быстрая загрузка нужных весов через aria2 (~15 сек)
print("Скачивание весов RMVPE, ContentVec и HiFi-GAN (40k)...")
models = [
    ("https://huggingface.co/IAHispano/Applio/resolve/main/Resources/predictors/rmvpe.pt", "rvc/models/predictors/rmvpe.pt"),
    ("https://huggingface.co/IAHispano/Applio/resolve/main/Resources/embedders/contentvec/pytorch_model.bin", "rvc/models/embedders/contentvec/pytorch_model.bin"),
    ("https://huggingface.co/IAHispano/Applio/resolve/main/Resources/embedders/contentvec/config.json", "rvc/models/embedders/contentvec/config.json"),
    ("https://huggingface.co/IAHispano/Applio/resolve/main/Resources/pretrained_v2/f0G40k.pth", "rvc/models/pretraineds/hifi-gan/f0G40k.pth"),
    ("https://huggingface.co/IAHispano/Applio/resolve/main/Resources/pretrained_v2/f0D40k.pth", "rvc/models/pretraineds/hifi-gan/f0D40k.pth"),
]

for url, out_path in models:
    if not os.path.exists(out_path):
        out_dir = os.path.dirname(out_path)
        out_name = os.path.basename(out_path)
        !aria2c -x 16 -s 16 -k 1M -q -d "{out_dir}" -o "{out_name}" "{url}"

!cp assets/config_template.json assets/config.json

import torch
print(f"Готово! GPU: {torch.cuda.get_device_name(0)}, PyTorch: {torch.__version__}")

## Шаг 2. Загрузка вашего чистого голоса и целевого трека

In [ ]:
import os, shutil

%cd /content
if not os.path.exists("/content/piper-voice"):
    !git clone --depth 1 https://github.com/Holki9/piper-voice.git /content/piper-voice
else:
    !cd /content/piper-voice && git pull origin main

shutil.copy2("/content/piper-voice/raw_audio/Testr2.wav", "/content/Applio/assets/datasets/MyVoice/Testr2.wav")
shutil.copy2("/content/piper-voice/songs/aty_baty.mp3", "/content/aty_baty.mp3")

print("Файлы проверены и скопированы:")
!ls -lh /content/Applio/assets/datasets/MyVoice/
!ls -lh /content/aty_baty.mp3

## Шаг 3. Предобработка и извлечение частот связок (RMVPE)

In [ ]:
%cd /content/Applio

# 1. Нарезка и ресемплинг
!python core.py preprocess --model-name MyVoice --dataset-path assets/datasets/MyVoice --sample-rate 40000 --cpu-cores 2 --clean

# 2. Извлечение питча (RMVPE) и контент-эмбеддингов (ContentVec) на GPU
!python core.py extract --model-name MyVoice --sample-rate 40000 --f0-method rmvpe --embedder-model contentvec

print("Шаг 3 завершен успешно!")

## Шаг 4. Обучение модели RVC v2 на вашем голосе (150 эпох, ~6-7 минут на GPU T4)

In [ ]:
%cd /content/Applio

# Запуск быстрого обучения на GPU T4 (150 эпох)
!python core.py train --model-name MyVoice --save-every-epoch 25 --total-epoch 150 --sample-rate 40000 --batch-size 4 --gpu 0 --vocoder HiFi-GAN --save-only-latest --pretrained

# Построение индекса поиска тембра
!python core.py index --model-name MyVoice --index-algorithm Auto

print("Шаг 4 завершен! Обученная модель:")
!ls -lh logs/MyVoice/

## Шаг 5. Спеть песню твоим голосом и скачать результат

In [ ]:
import glob, os, subprocess, shlex
from IPython.display import Audio, display
from google.colab import files

print("1. Разделение трека на вокал и инструментал (Demucs)...\n")
!demucs --two-stems=vocals /content/aty_baty.mp3 -o /content/separated

vocal_track = "/content/separated/htdemucs/aty_baty/vocals.wav"
no_vocal_track = "/content/separated/htdemucs/aty_baty/no_vocals.wav"

if not os.path.exists(vocal_track):
    vocals = glob.glob("/content/separated/**/vocals.wav", recursive=True)
    if vocals:
        vocal_track = vocals[0]
        no_vocal_track = vocal_track.replace("vocals.wav", "no_vocals.wav")

# Поиск путей обученной модели и индекса
!ls -t /content/Applio/logs/MyVoice/*.pth 2> /dev/null | head -n 1 > /tmp/pth.txt
pth_file = open("/tmp/pth.txt").read().strip()
if not pth_file:
    raise FileNotFoundError("Не найден файл обученной модели .pth в logs/MyVoice/")

!ls -t /content/Applio/logs/MyVoice/*.index 2> /dev/null | head -n 1 > /tmp/index.txt
index_file = open("/tmp/index.txt").read().strip()

print(f"Используем модель: {pth_file}")
print(f"Используем индекс: {index_file}")

print("2. Перепеваем вокальную дорожку твоим голосом (RVC v2)...\n")
%cd /content/Applio
infer_cmd = [
    "python", "core.py", "infer",
    "--pitch", "0",
    "--volume-envelope", "0.8",
    "--index-rate", "0.75",
    "--protect", "0.5",
    "--f0-method", "rmvpe",
    "--input-path", vocal_track,
    "--output-path", "/content/my_vocals.wav",
    "--pth-path", pth_file,
    "--export-format", "WAV",
    "--embedder-model", "contentvec"
]
if index_file and os.path.exists(index_file):
    infer_cmd += ["--index-path", index_file]

!{shlex.join(infer_cmd)}

print("3. Сведение твоего вокала с оригинальной аранжировкой...")
mix_cmd = [
    "ffmpeg", "-y",
    "-i", "/content/my_vocals.wav",
    "-i", no_vocal_track,
    "-filter_complex", "[0:a]volume=1.15[v];[1:a]volume=0.95[m];[v][m]amix=inputs=2:duration=first:dropout_transition=2",
    "-b:a", "320k",
    "/content/Трофимов_Аты_баты_Мой_голос.mp3"
]
subprocess.run(mix_cmd, check=True)

print("\n🎉 ГОТОВО! Песня Сергей Трофимов — Аты-баты перепета твоим голосом!")
display(Audio("/content/Трофимов_Аты_баты_Мой_голос.mp3"))

# Скачивание готовой песни и модели на устройство
print("Скачивание готового трека на компьютер...")
files.download("/content/Трофимов_Аты_баты_Мой_голос.mp3")
if pth_file and os.path.exists(pth_file):
    files.download(pth_file)
if index_file and os.path.exists(index_file):
    files.download(index_file)
